# Xanvora Video Factory — Free GPU Runner 🔥🤖

Generate real AI video with Wan2GP inside a Google Colab GPU session. The browser UI remains CPU-only; the actual generation runs on the Colab GPU.

**Free-tier target:** T4-class GPU / low-VRAM profile. The exact GPU is assigned by Colab and is not guaranteed.

**Important:** this notebook is an interactive generator, not a permanent public API server. Colab's free managed runtimes have dynamic resource limits and restrictions on persistent web services/workers. citehttps://research.google.com/colaboratory/faq.html


In [ ]:
import subprocess, os, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise RuntimeError('GPU not detected. In Colab choose Runtime → Change runtime type → GPU, then rerun.')


## 1. Install Wan2GP

This uses the official Wan2GP repository and its Python API.

In [ ]:
%cd /content
!rm -rf Wan2GP
!git clone --depth 1 https://github.com/deepbeepmeep/Wan2GP.git
%cd /content/Wan2GP
!pip -q install uv
!uv pip install --system -r requirements.txt
!apt-get -qq update && apt-get -qq install -y ffmpeg


## 2. Xanvora generation settings

On free Colab GPUs, start with a short 480p clip. The notebook discovers a compatible FastWan video model instead of hard-coding a model ID.

In [ ]:
PROMPT = '''Cinematic humanoid robot walking through a futuristic industrial laboratory, realistic materials, soft volumetric lighting, smooth camera tracking, subtle mechanical motion, premium robotics commercial.'''
DURATION_SECONDS = 5
RESOLUTION = '832x480'
STEPS = 8
FPS = 24

print('Prompt:', PROMPT)
print('Duration:', DURATION_SECONDS, 'seconds')
print('Resolution:', RESOLUTION)


In [ ]:
from pathlib import Path
import sys, os

ROOT = Path('/content/Wan2GP').resolve()
sys.path.insert(0, str(ROOT))

from shared.api import init

session = init(
    root=ROOT,
    cli_args=['--attention', 'sdpa', '--profile', '4'],
    console_output=True,
)

models = session.list_model_metadata(query='FastWan', main_output='video', limit=20)
if not models:
    models = session.list_model_metadata(main_output='video', limit=20)

print('Available video models:')
for m in models:
    print('-', m.get('model_type'), '|', m.get('name'))

MODEL_TYPE = models[0]['model_type']
print('\nSelected:', MODEL_TYPE)


## 3. Generate the real MP4 🎬

In [ ]:
import math

video_length = max(49, round(DURATION_SECONDS * FPS))

settings = {
    'model_type': MODEL_TYPE,
    'prompt': PROMPT,
    'resolution': RESOLUTION,
    'num_inference_steps': STEPS,
    'video_length': video_length,
    'duration_seconds': DURATION_SECONDS,
    'force_fps': FPS,
}

print('Submitting:', settings)
job = session.submit_task(settings)
result = job.result()

if not result.success or not result.generated_files:
    errors = [getattr(e, 'message', str(e)) for e in (result.errors or [])]
    raise RuntimeError('Generation failed: ' + ('; '.join(errors) if errors else 'unknown error'))

VIDEO_PATH = Path(result.generated_files[0])
print('SUCCESS:', VIDEO_PATH)


In [ ]:
from IPython.display import Video, display
from google.colab import files

display(Video(str(VIDEO_PATH), embed=True, width=832))
print('Download:', VIDEO_PATH)
files.download(str(VIDEO_PATH))


### Xanvora workflow

1. Open this notebook in Colab.
2. Enable a GPU runtime.
3. Run cells top-to-bottom.
4. Edit `PROMPT`, then rerun the generation cell for another clip.

Wan2GP's official API supports `init(...)`, `submit_task(...)`, progress/events, and generated output files. citehttps://github.com/deepbeepmeep/Wan2GP/blob/main/docs/API.md


## 4. Start the Xanvora Worker API 🌐
Run this cell after the successful test above. It starts a temporary HTTPS Worker endpoint backed by this Colab GPU. Keep the Colab session running while using the website.

In [ ]:
!pip -q install fastapi uvicorn[standard] pyngrok


In [ ]:
from pathlib import Path
worker_code = r'''from fastapi import FastAPI, HTTPException, Header
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from pathlib import Path
from uuid import uuid4
from threading import Lock
import shutil, os, sys, time, hmac

ROOT = Path('/content/Wan2GP').resolve()
OUT = ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
from shared.api import init

app = FastAPI(title='Xanvora Video Worker')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])
app.mount('/v1/video/files', StaticFiles(directory=str(OUT)), name='video-files')
WORKER_TOKEN = Path('/content/xanvora_worker_token.txt').read_text().strip() if Path('/content/xanvora_worker_token.txt').exists() else ''

session = init(root=ROOT, cli_args=['--attention','sdpa','--profile','4'], console_output=True)
models = session.list_model_metadata(query='FastWan', main_output='video', limit=20) or session.list_model_metadata(main_output='video', limit=20)
MODEL_TYPE = next((m['model_type'] for m in models if m.get('model_type') == 'ti2v_2_2_fastwan'), models[0]['model_type'])
jobs = {}
lock = Lock()

class JobRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=4000)
    engine: str = 'wan2gp'
    duration: int = Field(default=5, ge=4, le=10)

def auth(x_xanvora_token: str = Header(default='')):
    if not WORKER_TOKEN or not hmac.compare_digest(x_xanvora_token, WORKER_TOKEN):
        raise HTTPException(401, 'invalid worker token')

@app.get('/health')
def health(x_xanvora_token: str = Header(default='')):
    auth(x_xanvora_token)
    return {'ok': True, 'model_type': MODEL_TYPE, 'busy': lock.locked()}

def run_job(job_id, req):
    with lock:
        try:
            jobs[job_id]['status'] = 'generating'
            fps = 24
            settings = {'model_type': MODEL_TYPE, 'prompt': req.prompt, 'resolution':'832x480', 'num_inference_steps':8, 'video_length':max(49, round(req.duration*fps)), 'duration_seconds':req.duration, 'force_fps':fps}
            result = session.submit_task(settings).result()
            if not result.success or not result.generated_files:
                errors = [getattr(e,'message',str(e)) for e in (result.errors or [])]
                raise RuntimeError('; '.join(errors) if errors else 'generation failed')
            src = Path(result.generated_files[0])
            dest = OUT / f'{job_id}{src.suffix}'
            shutil.copy2(src, dest)
            jobs[job_id].update(status='completed', video_url=f'/v1/video/files/{dest.name}')
        except Exception as exc:
            jobs[job_id].update(status='error', error=str(exc))

@app.post('/v1/video/jobs', status_code=202)
def create_job(req: JobRequest, x_xanvora_token: str = Header(default='')):
    auth(x_xanvora_token)
    if lock.locked():
        raise HTTPException(409, 'GPU worker is busy; retry after the current job completes')
    job_id = uuid4().hex
    jobs[job_id] = {'id':job_id, 'status':'queued', 'engine':req.engine, 'prompt':req.prompt, 'duration':req.duration, 'created_at':time.time()}
    import threading
    threading.Thread(target=run_job, args=(job_id,req), daemon=True).start()
    return jobs[job_id]

@app.get('/v1/video/jobs/{job_id}')
def get_job(job_id: str, x_xanvora_token: str = Header(default='')):
    auth(x_xanvora_token)
    if job_id not in jobs: raise HTTPException(404, 'job not found')
    return jobs[job_id]
'''
Path('/content/xanvora_worker.py').write_text(worker_code)
print('Worker written to /content/xanvora_worker.py')



In [ ]:
import subprocess, time, secrets, re
token = secrets.token_urlsafe(24)
Path('/content/xanvora_worker_token.txt').write_text(token)
try: subprocess.run(['pkill','-f','uvicorn.*xanvora_worker'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception: pass
proc = subprocess.Popen(['uvicorn','xanvora_worker:app','--host','0.0.0.0','--port','8000'], cwd='/content', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(4)
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(5)
public_url=None
for _ in range(30):
    line=tunnel.stdout.readline().strip()
    if line: print(line)
    m=re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', line)
    if m: public_url=m.group(0); break
print('\nXANVORA_WORKER_URL=' + str(public_url))
print('XANVORA_WORKER_TOKEN=' + token)
print('Keep this Colab session running while Xanvora generates videos.')


### 5. Connect the website
Open Xanvora Video Factory, paste the printed `XANVORA_WORKER_URL` into **Worker URL**, then press **Generate**. Keep this Colab tab/session running until the video finishes.

In [ ]:
import requests
print(requests.get('http://127.0.0.1:8000/health').json())
